In [6]:
import neo4j

import pandas as pd

from IPython.display import display

import numpy as np
import requests
from io import StringIO

## Web server interface at https://xxxx:7473

#### Update - since the videos were filmed, neo4j requires a longer, more complex password, so the newest password is here:

**Username: neo4j**

**Password: ucb_mids_w205**

**In the web server interface, run the same query from last week to return all nodes and all relationships:**

```
match (n) return n
```

In [16]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))
session = driver.session(database="neo4j")

In [17]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

def my_neo4j_nodes_relationships():
    "print all the nodes and relationships"
   
    print("-------------------------")
    print("  Nodes:")
    print("-------------------------")
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    display(df)
    
    print("-------------------------")
    print("  Relationships:")
    print("-------------------------")
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    display(df)
    
    density = (2 * number_relationships) / (number_nodes * (number_nodes - 1))
    
    print("-------------------------")
    print("  Density:", f'{density:.1f}')
    print("-------------------------")
    

In [15]:
# Load Data
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-08-22/population.csv"

# Read the data directly into a pandas DataFrame
refugee = pd.read_csv(url)
refugee.head()

,year,coo_name,coo,coo_iso,coa_name,coa,coa_iso,refugees,asylum_seekers,returned_refugees,idps,returned_idps,stateless,ooc,oip,hst
0,2010,Afghanistan,AFG,AFG,Afghanistan,AFG,AFG,0,0,0,351907,3366,0,838250,NaN,NaN
1,2010,Iran (Islamic Rep. of),IRN,IRN,Afghanistan,AFG,AFG,30,21,0,0,0,0,0,NaN,NaN
2,2010,Iraq,IRQ,IRQ,Afghanistan,AFG,AFG,6,0,0,0,0,0,0,NaN,NaN
3,2010,Pakistan,PAK,PAK,Afghanistan,AFG,AFG,6398,9,0,0,0,0,0,NaN,NaN
4,2010,Egypt,ARE,EGY,Albania,ALB,ALB,5,0,0,0,0,0,0,NaN,NaN


# Create Graphs 

In [18]:
def create_graph_1(tx, refugee_df):
    """Graph 1: Origin -> Asylum (refugees flowing out)"""
    for _, row in refugee_df.iterrows():
        origin = row['coo_name']
        asylum = row['coa_name']
        num_refugees = row['refugees']
        
        query = """
        MERGE (origin:Country {name: $origin})
        MERGE (asylum:Country {name: $asylum})
        MERGE (origin)-[r:REFUGEES_TO]->(asylum)
        ON CREATE SET r.count = $num_refugees
        ON MATCH SET r.count = r.count + $num_refugees
        """
        tx.run(query, origin=origin, asylum=asylum, num_refugees=num_refugees)

def create_graph_2(tx, refugee_df):
    """Graph 2: Asylum -> Origin (refugees flowing in)"""
    for _, row in refugee_df.iterrows():
        origin = row['coo_name']
        asylum = row['coa_name']
        num_refugees = row['refugees']
        
        query = """
        MERGE (origin:Country {name: $origin})
        MERGE (asylum:Country {name: $asylum})
        MERGE (asylum)-[r:RECEIVES_FROM]->(origin)
        ON CREATE SET r.count = $num_refugees
        ON MATCH SET r.count = r.count + $num_refugees
        """
        tx.run(query, origin=origin, asylum=asylum, num_refugees=num_refugees)

In [11]:
refugee['coo_name'].unique()

array(['Afghanistan', 'Iran (Islamic Rep. of)', 'Iraq', 'Pakistan',
       'Egypt', 'China', 'Palestinian',
       'Serbia and Kosovo: S/RES/1244 (1999)', 'Türkiye', 'Angola',
       'Benin', 'Chad', 'Cameroon', 'Congo', 'Dem. Rep. of the Congo',
       'Guinea', "Cote d'Ivoire", 'Liberia', 'Libya', 'Niger', 'Nigeria',
       'Somalia', 'Sudan', 'Western Sahara', 'Burundi',
       'Central African Rep.', 'Eritrea', 'Ethiopia', 'Guinea-Bissau',
       'Mauritania', 'Rwanda', 'Senegal', 'Sierra Leone',
       'United Rep. of Tanzania', 'Unknown', 'Algeria', 'Djibouti',
       'Kazakhstan', 'Mali', 'Russian Federation', 'Saudi Arabia',
       'Syrian Arab Rep.', 'Tajikistan', 'Turkmenistan', 'Tunisia',
       'Uganda', 'Uzbekistan', 'Yemen', 'Zimbabwe', 'Stateless',
       'Albania', 'Armenia', 'Bangladesh',
       'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina',
       'Chile', 'Colombia', 'Costa Rica', 'Cuba', 'Dominican Rep.',
       'Ecuador', 'Georgia', 'Ghana', 'Haiti',

In [19]:
# RUN GRAPHS
my_neo4j_wipe_out_database()
session.execute_write(create_graph_1, refugee)

In [13]:
my_neo4j_nodes_relationships()

-------------------------
  Nodes:
-------------------------


,node_name,labels
0,Afghanistan,[Country]
1,Albania,[Country]
2,Algeria,[Country]
3,Anchorage,[Station]
4,Andorra,[Country]
...,...,...
221,Washington,[Station]
222,Western Sahara,[Country]
223,Yemen,[Country]
224,Zambia,[Country]


-------------------------
  Relationships:
-------------------------


,node_name_1,node_1_labels,relationship_type,node_name_2,node_2_labels
0,Afghanistan,[Country],REFUGEES_TO,Afghanistan,[Country]
1,Afghanistan,[Country],REFUGEES_TO,Albania,[Country]
2,Afghanistan,[Country],REFUGEES_TO,Algeria,[Country]
3,Afghanistan,[Country],REFUGEES_TO,Argentina,[Country]
4,Afghanistan,[Country],REFUGEES_TO,Armenia,[Country]
...,...,...,...,...,...
7743,Zimbabwe,[Country],REFUGEES_TO,Ukraine,[Country]
7744,Zimbabwe,[Country],REFUGEES_TO,United Kingdom of Great Britain and Northern I...,[Country]
7745,Zimbabwe,[Country],REFUGEES_TO,United States of America,[Country]
7746,Zimbabwe,[Country],REFUGEES_TO,Zambia,[Country]


-------------------------
  Density: 0.3
-------------------------


In [64]:
# 1. CLEAN & AGGREGATE (Kenneth's Logic)
# Grouping first makes the database import significantly faster
my_neo4j_wipe_out_database()
aggregated = refugee.groupby(
    ["coo_name", "coo_iso", "coa_name", "coa_iso"],
    as_index=False
)["refugees"].sum()

# Remove self-loops with 0 weight and all 0-weight records
aggregated = aggregated[~((aggregated["coo_iso"] == aggregated["coa_iso"]) & (aggregated["refugees"] == 0))]
aggregated = aggregated[aggregated["refugees"] > 0]

# 2. CREATE UNIQUE NODES (Using ISO codes for reliability)
origin_countries = aggregated[["coo_name", "coo_iso"]].rename(columns={"coo_name": "name", "coo_iso": "iso"})
asylum_countries = aggregated[["coa_name", "coa_iso"]].rename(columns={"coa_name": "name", "coa_iso": "iso"})
all_countries = pd.concat([origin_countries, asylum_countries]).drop_duplicates(subset="iso")

for _, row in all_countries.iterrows():
    query = """
        MERGE (c:Country {iso: $iso})
        SET c.name = $name
    """
    session.run(query, iso=row["iso"], name=row["name"])

# 3. CREATE THE TWO GRAPHS (Relationships)
for _, row in aggregated.iterrows():
    query = """
        MATCH (origin:Country {iso: $coo_iso})
        MATCH (asylum:Country {iso: $coa_iso})
        
        // Graph 1: Emigrating FROM -> Immigrating TO
        MERGE (origin)-[out:REFUGEES_TO]->(asylum)
        SET out.weight = $weight
        
        // Graph 2: Immigrating TO -> Emigrating FROM
        MERGE (asylum)-[inc:RECEIVES_FROM]->(origin)
        SET inc.weight = $weight
    """
    session.run(query, 
                coo_iso=row["coo_iso"], 
                coa_iso=row["coa_iso"], 
                weight=int(row["refugees"]))

print("Both graphs created successfully.")

# check number of refugees going from country to another

country_name = "Ukraine" 

query = """
MATCH (origin:Country {name: $name})-[r:REFUGEES_TO]->(asylum:Country)
RETURN asylum.name AS Destination, 
       r.weight AS Number_of_Refugees
ORDER BY Number_of_Refugees DESC
"""

df_specific = my_neo4j_run_query_pandas(query, name=country_name)
display(df_specific)

Both graphs created successfully.


,Destination,Number_of_Refugees
0,Russian Federation,2311259
1,Germany,1067645
2,Poland,960602
3,Czechia,436660
4,United Kingdom of Great Britain and Northern I...,184407
...,...,...
63,Dominican Rep.,10
64,Paraguay,9
65,Colombia,8
66,Thailand,5


In [25]:
my_neo4j_nodes_relationships()

-------------------------
  Nodes:
-------------------------


,node_name,labels
0,Afghanistan,[Country]
1,Albania,[Country]
2,Algeria,[Country]
3,Andorra,[Country]
4,Angola,[Country]
...,...,...
200,Viet Nam,[Country]
201,Western Sahara,[Country]
202,Yemen,[Country]
203,Zambia,[Country]


-------------------------
  Relationships:
-------------------------


,node_name_1,node_1_labels,relationship_type,node_name_2,node_2_labels
0,Afghanistan,[Country],DISPLACED_TO,Albania,[Country]
1,Afghanistan,[Country],REFUGEES_TO,Albania,[Country]
2,Afghanistan,[Country],DISPLACED_TO,Argentina,[Country]
3,Afghanistan,[Country],REFUGEES_TO,Argentina,[Country]
4,Afghanistan,[Country],DISPLACED_TO,Armenia,[Country]
...,...,...,...,...,...
16957,Zimbabwe,[Country],REFUGEES_TO,United Kingdom of Great Britain and Northern I...,[Country]
16958,Zimbabwe,[Country],DISPLACED_TO,United States of America,[Country]
16959,Zimbabwe,[Country],REFUGEES_TO,United States of America,[Country]
16960,Zimbabwe,[Country],DISPLACED_TO,Zambia,[Country]


-------------------------
  Density: 0.8
-------------------------


# Algorithms: PageRank and Minimum Spanning Tree

## PageRank
1. Run PageRank on 1st graph: returns most influential countries in granting asylum
2. Run PageRank on 2nd graph: returns most influential countries in creating refugees

In [66]:
session.run("CALL gds.graph.drop('graph_outflow', false)")
session.run("CALL gds.graph.drop('graph_inflow', false)")

# Project Graph 1: Origin -> Asylum
session.run("""
    CALL gds.graph.project(
      'graph_outflow',
      'Country',
      'REFUGEES_TO',
      { relationshipProperties: 'weight' }
    )
""")

# Project Graph 2: Asylum -> Origin
session.run("""
    CALL gds.graph.project(
      'graph_inflow',
      'Country',
      'RECEIVES_FROM',
      { relationshipProperties: 'weight' }
    )
""")
print("Graphs projected into GDS.")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL gds.graph.drop('graph_outflow', false)"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL gds.graph.drop('graph_inflow', false)"


Graphs projected into GDS.


In [67]:
def get_pagerank_query(graph_name, limit=10):
    """
    graph_name='graph_outflow':  Finds top Asylum Granters
    graph_name='graph_inflow' :  Finds top Refugee Creators
    """
    return f"""
    CALL gds.pageRank.stream('{graph_name}', {{
        relationshipWeightProperty: 'weight'
    }})
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS Country, 
           gds.util.asNode(nodeId).iso AS iso,
           score
    ORDER BY score DESC 
    LIMIT {limit}
    """

In [68]:
# 1. PageRank on graph 1: return most influential countries in granting asylum
df_granters = my_neo4j_run_query_pandas(get_pagerank_query('graph_outflow'))
df_granters

,Country,iso,score
0,Canada,CAN,49.782578
1,United States of America,USA,43.325046
2,Germany,DEU,28.820270
3,United Kingdom of Great Britain and Northern I...,GBR,6.681935
4,France,FRA,4.549955
5,Australia,AUS,3.145572
6,Sweden,SWE,2.863837
7,Rep. of Moldova,MDA,2.205823
8,Russian Federation,RUS,1.934069
9,Switzerland,CHE,1.672228


In [69]:
# 2. PageRank on graph 2: return most influential countries in creating refugees
df_creators = my_neo4j_run_query_pandas(get_pagerank_query('graph_inflow'))
df_creators

,Country,iso,score
0,Afghanistan,AFG,16.324892
1,Pakistan,PAK,14.733361
2,Syrian Arab Rep.,SYR,11.652782
3,Iraq,IRQ,11.009828
4,Dem. Rep. of the Congo,COD,6.723109
5,Somalia,SOM,6.359195
6,Sudan,SDN,5.292563
7,South Sudan,SSD,5.176915
8,Yemen,YEM,3.740305
9,Colombia,COL,3.201706


## Minimum Spanning Tree
1. Select top X asylum granters (from pagerank 1). For each top asylum granter, run minimum spanning tree on the 2nd graph to see how refugees flow from the top refugee creators

2. Select top X refugee creators (from pagerank 2). For each top refugee creator, run minimum spanning tree on 1st graph to see how refugees flow from top asylum granters

In [70]:
def get_projection_queries():
    """
    Updated projections to use UNDIRECTED orientation. 
    This is required for the Spanning Tree algorithm.
    """
    q1 = """
    CALL gds.graph.project(
      'graph_outflow',
      'Country',
      {
        REFUGEES_TO: {
          orientation: 'UNDIRECTED',
          properties: 'weight'
        }
      }
    )
    """
    q2 = """
    CALL gds.graph.project(
      'graph_inflow',
      'Country',
      {
        RECEIVES_FROM: {
          orientation: 'UNDIRECTED',
          properties: 'weight'
        }
      }
    )
    """
    return [q1, q2]

In [79]:
def get_mst_query(graph_name):
    return f"""
    MATCH (n:Country {{iso: $iso}})
    CALL gds.spanningTree.stream('{graph_name}', {{
        sourceNode: n,
        relationshipWeightProperty: 'weight',
        objective: 'maximum'
    }})
    YIELD nodeId, parentId, weight
    WHERE nodeId <> parentId
    // ONLY return paths where the source or destination is our target country
    WITH gds.util.asNode(parentId) AS p, gds.util.asNode(nodeId) AS c, weight
    WHERE p.iso = $iso OR c.iso = $iso
    RETURN p.name AS Source, c.name AS Destination, weight
    ORDER BY weight DESC
    """

In [80]:
# rerun projection for mst
# Define the names clearly
graph_names = ['graph_outflow', 'graph_inflow']

# 1. Drop old graphs first (to avoid "Graph already exists" errors)
for name in graph_names:
    drop_query = f"CALL gds.graph.drop('{name}', false)"
    my_neo4j_run_query_pandas(drop_query)

# 2. Project the new "Undirected" versions
for q in get_projection_queries():
    my_neo4j_run_query_pandas(q)

print("Old graphs dropped and new undirected graphs projected successfully.")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL gds.graph.drop('graph_outflow', false)"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated field from a procedure. ('schema' returned by 'gds.graph.drop' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL gds.graph.drop('graph_inflow', false)"


Old graphs dropped and new undirected graphs projected successfully.


In [81]:
# 1. select top 10 asylum granters from pagerank 1. For each granter, run MST on graph 2
# --- LOOP 1: Top Asylum Granters -> MST on Graph 2 (Inflow) ---
print("### ANALYZING TOP ASYLUM GRANTERS (Inflow Backbone)")
for index, row in df_granters.head().iterrows():
    country_name = row['Country']
    country_iso = row['iso']
    
    print(f"\n--- MST for Granter: {country_name} ({country_iso}) ---")
    # We run the MST on 'graph_inflow' to see how people arrived there
    mst_df = my_neo4j_run_query_pandas(get_mst_query('graph_inflow'), iso=country_iso)
    display(mst_df)

### ANALYZING TOP ASYLUM GRANTERS (Inflow Backbone)

--- MST for Granter: Canada (CAN) ---


,Source,Destination,weight
0,Canada,Colombia,141713.0
1,Canada,Saint Vincent and the Grenadines,14134.0
2,Canada,Jamaica,9016.0
3,Canada,Saint Lucia,8787.0
4,Canada,Guyana,4260.0
5,Canada,Bahamas,2469.0
6,Canada,Grenada,2020.0
7,Canada,Barbados,1784.0
8,Canada,Antigua and Barbuda,838.0
9,Canada,Saint Kitts and Nevis,326.0



--- MST for Granter: United States of America (USA) ---


,Source,Destination,weight
0,United States of America,China,896441.0
1,United States of America,Haiti,252705.0
2,United States of America,El Salvador,210684.0
3,United States of America,Guatemala,170464.0
4,United States of America,Honduras,110651.0
5,United States of America,Nepal,79135.0
6,United States of America,Cuba,15381.0
7,United States of America,Kyrgyzstan,14156.0
8,United States of America,Mongolia,10119.0
9,United States of America,Kazakhstan,9867.0



--- MST for Granter: Germany (DEU) ---


,Source,Destination,weight
0,Germany,Syrian Arab Rep.,3943021.0
1,Germany,Afghanistan,1079892.0
2,Germany,Ukraine,1067645.0
3,Germany,Serbia and Kosovo: S/RES/1244 (1999),429888.0
4,Germany,Unknown,274156.0
5,Germany,North Macedonia,23759.0
6,Germany,Palau,5.0



--- MST for Granter: United Kingdom of Great Britain and Northern Ireland (GBR) ---


,Source,Destination,weight
0,United Kingdom of Great Britain and Northern I...,Unknown,249838.0
1,United Kingdom of Great Britain and Northern I...,Zimbabwe,100712.0
2,United Kingdom of Great Britain and Northern I...,Kuwait,8162.0
3,United Kingdom of Great Britain and Northern I...,Dem. People's Rep. of Korea,4927.0
4,United Kingdom of Great Britain and Northern I...,Mauritius,539.0
5,United Kingdom of Great Britain and Northern I...,Maldives,323.0



--- MST for Granter: France (FRA) ---


,Source,Destination,weight
0,France,Unknown,774800.0
1,France,Sri Lanka,327177.0
2,France,Cambodia,152716.0
3,France,Guinea,117209.0
4,France,Lao People's Dem. Rep.,89331.0
5,France,Albania,58140.0
6,France,Georgia,43419.0
7,France,Comoros,9189.0
8,France,Madagascar,3302.0


In [82]:
# 2. select top 10 refugee creators from pagerank 2. For each creator, run MST on graph 1
# --- LOOP 2: Top Refugee Creators -> MST on Graph 1 (Outflow) ---
print("\n\n### ANALYZING TOP REFUGEE CREATORS (Outflow Backbone)")
for index, row in df_creators.iterrows():
    country_name = row['Country']
    country_iso = row['iso']
    
    print(f"\n--- MST for Creator: {country_name} ({country_iso}) ---")
    # We run the MST on 'graph_outflow' to see where they were displaced to
    mst_df = my_neo4j_run_query_pandas(get_mst_query('graph_outflow'), iso=country_iso)
    display(mst_df)



### ANALYZING TOP REFUGEE CREATORS (Outflow Backbone)

--- MST for Creator: Afghanistan (AFG) ---


,Source,Destination,weight
0,Afghanistan,Pakistan,20160554.0
1,Afghanistan,Iran (Islamic Rep. of),14183143.0
2,Afghanistan,Germany,1079892.0
3,Afghanistan,Australia,113114.0
4,Afghanistan,Tajikistan,51073.0
5,Afghanistan,Indonesia,46707.0
6,Afghanistan,Uzbekistan,27198.0



--- MST for Creator: Pakistan (PAK) ---


,Source,Destination,weight
0,Pakistan,Afghanistan,20160554.0



--- MST for Creator: Syrian Arab Rep. (SYR) ---


...

,Source,Destination,weight
0,South Sudan,Uganda,6417397.0
1,South Sudan,Sudan,5379885.0
2,South Sudan,Ethiopia,3341018.0



--- MST for Creator: Yemen (YEM) ---


,Source,Destination,weight
0,Yemen,Somalia,2641678.0



--- MST for Creator: Colombia (COL) ---


,Source,Destination,weight
0,Colombia,Venezuela (Bolivarian Republic of),1724081.0
1,Colombia,Ecuador,1246134.0
2,Colombia,Canada,141713.0
3,Colombia,Panama,119556.0
4,Colombia,Costa Rica,82473.0
5,Colombia,Chile,14101.0
6,Colombia,Uruguay,1724.0
